# Risk scoring + Explainability + Random Forest

Owner: Thanthrige D.P

Put `PhiUSIIL_Phishing_URL_Dataset.csv` in `data/raw/` before running.

## To do

- [ ] Train, tune and evaluate the Random Forest in `src/models/forest.py`
- [ ] Risk score from the phishing probability, risk bands (4.11)
- [ ] Calibration check: calibration curve / Brier score before and after
- [ ] SHAP global importance and summary plot (4.13)
- [ ] Explain single predictions: score + class + top features
- [ ] Final analysis across all parts (4.14)

## Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src import config, data, features, evaluate, risk, explain
from src.models import logreg, tree, svm, forest

# Work on a sample first; set SAMPLE = None for the full dataset.
SAMPLE = 20000

df = data.load_clean_data()
if SAMPLE and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True)
df.shape

## Phase 1: Random Forest (proposal 4.9)

Cross-validation on the training set only (`evaluate.cv_splitter()`, 5 stratified folds).
The test set is not touched here.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")  # harmless joblib notice
from sklearn.ensemble import RandomForestClassifier

original = data.get_original_features(df)
df = features.add_features(df)
candidates = original + features.ENGINEERED_FEATURES
X_train_all, _, _, _ = data.split(df, candidates)
# Feature selection sees training rows only.
engineered = features.select_features(df.loc[X_train_all.index], candidates)
feature_sets = {"original": original, "engineered": engineered}
{name: len(cols) for name, cols in feature_sets.items()}

In [ ]:
SHOW = ["f1_mean", "f1_std", "roc_auc_mean", "recall_mean", "precision_mean", "accuracy_mean"]

def cv_table(make_model):
    """CV metrics of make_model() on each feature set (training rows only)."""
    rows = {}
    for name, cols in feature_sets.items():
        X_tr, _, y_tr, _ = data.split(df, cols)
        rows[name] = evaluate.cv_evaluate(make_model(), X_tr, y_tr)
    return pd.DataFrame(rows).T[SHOW]

def default_forest():
    """Baseline: sklearn defaults with 200 trees (the starting build())."""
    return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=config.RANDOM_STATE)

baseline_cv = cv_table(default_forest)
baseline_cv.round(4)

### Hyperparameter search

`forest.tune()`: RandomizedSearchCV, 20 candidates, shared CV folds, chosen by F1,
on the original-feature training set. One set of values goes into `build()`,
because `main.py` uses the same `build()` for both feature sets.

In [ ]:
X_train, X_test, y_train, y_test = data.split(df, original)
search, seconds = forest.tune(X_train, y_train, n_iter=20, scoring="f1")
print(f"Search took {seconds/60:.1f} min for {len(search.cv_results_['params'])} candidates x {config.CV_FOLDS} folds")

# Full search results to a file; only the top rows are shown here.
out = config.RESULTS_DIR / (f"results_sample_{SAMPLE}_forest_search.csv" if SAMPLE else "forest_search.csv")
pd.DataFrame(search.cv_results_).to_csv(out, index=False)
print("best:", search.best_params_)
forest.search_table(search, top=8)

### Tuned model vs baseline

The search is flat: six candidates tie for best CV F1 (0.99993, std 0.00015), and every
other candidate is one URL worse over all folds, which is within noise. `best_params_`
just reports the first of the tied six. Tie-break rule: take the cheapest and most
regularised tied candidate (100 trees, max_depth 30, min_samples_leaf 8, sqrt features),
since extra trees buy nothing here and larger leaves give smoother probabilities for the
risk score. These values are written into `forest.build()`. Provisional until the
leakage check (phase 2): scores this close to 1.0 are a warning, not a result.


In [ ]:
tuned_cv = cv_table(forest.build)
comparison = pd.concat({"baseline": baseline_cv, "tuned": tuned_cv}).round(4)
comparison

## Final test-set report

Run once, after tuning is finished. Do not choose models or hyperparameters from this result.

In [ ]:
# Phase 5 only: run once on the full data after all choices are made.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    model = forest.train(X_train, y_train)
    display(evaluate.evaluate(model, X_test, y_test))

## Experiment log

| Date | What changed | Why | CV or test | Key metrics (F1 / ROC-AUC / recall) | Commit |
|---|---|---|---|---|---|
| 2026-11-17 | Baseline RF (200 trees, sklearn defaults), original (50) and engineered (52) features | Starting point (4.9) | CV, 5 folds, 20k sample | orig 0.9999 / 1.0000 / 0.9997; eng 0.9999 / 1.0000 / 0.9999 | phase 1 |
| 2026-11-17 | RandomizedSearchCV, 20 candidates, scored on F1, original features | Tune RF hyperparameters on train only | CV, 20k sample | 6-way tie at F1 0.99993; all 20 within 0.0001; 1.5-3.1 min | phase 1 |
| 2026-11-17 | build(): 100 trees, max_depth 30, min_samples_leaf 8, sqrt | Cheapest, most regularised tied candidate | CV, 20k sample | orig 0.9999 / 1.0000 / 0.9999; eng 0.9999 / 1.0000 / 0.9997; 1 CV error in 16,000 | phase 1 |
